In [1]:
# Install Dependencies
!pip install pandas scikit-learn matplotlib seaborn streamlit pyngrok

In [10]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
SPOTIFY_GREEN = '#1DB954'
SPOTIFY_BLACK = '#191414'
SPOTIFY_WHITE = '#FFFFFF'
PALETTE = px.colors.qualitative.Vivid

In [7]:
# Load 3 Datasets
songs = pd.read_csv("Spotify Wrapped Top 50 Songs 2025.csv")
artists = pd.read_csv("Spotify Wrapped Top 50 Artists 2025.csv")
alltime = pd.read_csv("Spotify Alltime Top 100 Songs.csv")
df = pd.concat([songs, artists, alltime], ignore_index=True)
songs.columns   = songs.columns.str.strip()
artists.columns = artists.columns.str.strip()
alltime.columns = alltime.columns.str.strip()
print(f'Songs    shape : {songs.shape}')
print(f'Artists  shape : {artists.shape}')
print(f'All-Time shape : {alltime.shape}')
songs.head(10)

Songs    shape : (50, 16)
Artists  shape : (50, 11)
All-Time shape : (100, 14)


,wrapped_2025_rank,song_title,artist,streams_2025_billions,primary_genre,bpm,duration_seconds,release_year,artist_country,explicit,danceability,energy,valence,acousticness,peak_global_chart_position,dataset_part
0,1,Die With A Smile,Lady Gaga & Bruno Mars,1.70,Pop,120,251,2024,USA,False,0.72,0.68,0.55,0.18,1,Spotify Wrapped 2025 Top 50 Global Songs
1,2,APT.,ROSE & Bruno Mars,1.62,K-Pop/Pop,145,173,2024,South Korea,False,0.86,0.79,0.88,0.05,1,Spotify Wrapped 2025 Top 50 Global Songs
2,3,Espresso,Sabrina Carpenter,1.55,Pop,104,175,2024,USA,False,0.80,0.73,0.96,0.06,1,Spotify Wrapped 2025 Top 50 Global Songs
3,4,Please Please Please,Sabrina Carpenter,1.48,Pop,107,186,2024,USA,False,0.70,0.66,0.87,0.10,1,Spotify Wrapped 2025 Top 50 Global Songs
4,5,Taste,Sabrina Carpenter,1.41,Pop,117,177,2024,USA,True,0.78,0.70,0.82,0.08,1,Spotify Wrapped 2025 Top 50 Global Songs
5,6,Birds Of A Feather,Billie Eilish,1.38,Indie Pop,88,210,2024,USA,False,0.75,0.44,0.65,0.38,1,Spotify Wrapped 2025 Top 50 Global Songs
6,7,Not Like Us,Kendrick Lamar,1.35,Hip-Hop,101,274,2024,USA,True,0.82,0.72,0.62,0.04,1,Spotify Wrapped 2025 Top 50 Global Songs
7,8,"Good Luck, Babe!",Chappell Roan,1.29,Pop,131,219,2024,USA,False,0.66,0.78,0.56,0.04,1,Spotify Wrapped 2025 Top 50 Global Songs
8,9,Beautiful Things,Benson Boone,1.24,Pop Rock,97,211,2024,USA,False,0.46,0.68,0.46,0.07,1,Spotify Wrapped 2025 Top 50 Global Songs
9,10,Lose Control,Teddy Swims,1.20,R&B/Soul,90,202,2023,USA,False,0.60,0.55,0.64,0.21,1,Spotify Wrapped 2025 Top 50 Global Songs


In [8]:
# Data Preprocessing & Feature Engineering
songs_std = songs.rename(columns={
    'streams_2025_billions' : 'streams',
    'wrapped_2025_rank'     : 'rank'
}).copy()
alltime_std = alltime.rename(columns={
    'total_streams_billions': 'streams',
    'alltime_rank'          : 'rank'
}).copy()

songs_std['source']   = 'Wrapped 2025'
alltime_std['source'] = 'All-Time Top 100'
COMMON = ['song_title', 'artist', 'streams', 'primary_genre',
          'bpm', 'release_year', 'artist_country', 'explicit',
          'danceability', 'energy', 'valence', 'acousticness', 'source']
df = pd.concat(
    [songs_std[COMMON], alltime_std[COMMON]],
    ignore_index=True
)
df = df.drop_duplicates(subset=['song_title', 'artist']).reset_index(drop=True)
df['bpm_norm'] = (df['bpm'] - df['bpm'].min()) / (df['bpm'].max() - df['bpm'].min())
df['popularity'] = (df['streams'] - df['streams'].min()) / (df['streams'].max() - df['streams'].min())
def broad_genre(g):
    g = str(g).lower()
    if 'pop' in g and 'k-pop' not in g:    return 'Pop'
    if 'k-pop' in g or 'k pop' in g:       return 'K-Pop'
    if 'hip' in g or 'rap' in g:           return 'Hip-Hop'
    if 'r&b' in g or 'soul' in g:          return 'R&B/Soul'
    if 'rock' in g or 'alt' in g:          return 'Rock/Alt'
    if 'country' in g:                      return 'Country'
    if 'reggaeton' in g or 'latin' in g:   return 'Latin'
    if 'edm' in g or 'dance' in g:         return 'EDM'
    if 'indie' in g or 'folk' in g:        return 'Indie/Folk'
    if 'synth' in g or 'electro' in g:     return 'Synth/Electronic'
    return 'Other'
df['genre_group'] = df['primary_genre'].apply(broad_genre)
print(f'Total songs in pool : {len(df)}')
print(f'Missing values      :\n{df[["danceability","energy","valence","acousticness","bpm"]].isnull().sum()}')
print(f'\nGenre distribution  :\n{df["genre_group"].value_counts()}')
df.head(5)

Total songs in pool : 135
Missing values      :
danceability    0
energy          0
valence         0
acousticness    0
bpm             0
dtype: int64

Genre distribution  :
genre_group
Pop           94
Rock/Alt      12
Hip-Hop       11
Other          4
K-Pop          3
Latin          3
R&B/Soul       3
Country        3
Indie/Folk     2
Name: count, dtype: int64


,song_title,artist,streams,primary_genre,bpm,release_year,artist_country,explicit,danceability,energy,valence,acousticness,source,bpm_norm,popularity,genre_group
0,Die With A Smile,Lady Gaga & Bruno Mars,1.70,Pop,120,2024,USA,False,0.72,0.68,0.55,0.18,Wrapped 2025,0.449612,0.242553,Pop
1,APT.,ROSE & Bruno Mars,1.62,K-Pop/Pop,145,2024,South Korea,False,0.86,0.79,0.88,0.05,Wrapped 2025,0.643411,0.225532,K-Pop
2,Espresso,Sabrina Carpenter,1.55,Pop,104,2024,USA,False,0.80,0.73,0.96,0.06,Wrapped 2025,0.325581,0.210638,Pop
3,Please Please Please,Sabrina Carpenter,1.48,Pop,107,2024,USA,False,0.70,0.66,0.87,0.10,Wrapped 2025,0.348837,0.195745,Pop
4,Taste,Sabrina Carpenter,1.41,Pop,117,2024,USA,True,0.78,0.70,0.82,0.08,Wrapped 2025,0.426357,0.180851,Pop


In [12]:
# Exploratory Data Analysis
# Audio Feature Correlation Heatmap
FEATURES = ['danceability', 'energy', 'valence', 'acousticness', 'bpm_norm', 'popularity']
corr = df[FEATURES].corr().round(2)
fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.columns,
    colorscale=[[0,'#191414'],[0.5,'#535353'],[1,'#1DB954']],
    text=corr.values,
    texttemplate='%{text}',
    showscale=True,
    zmin=-1, zmax=1
))
fig.update_layout(
    title=dict(text='🔥 Audio Feature Correlation Heatmap', font_size=18),
    height=480,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor=SPOTIFY_BLACK,
    font=dict(color=SPOTIFY_WHITE, family='Arial')
)
fig.show()

In [13]:
# Average Audio Features by Genre Group
genre_avg = df.groupby('genre_group')[['danceability','energy','valence','acousticness']].mean().reset_index()
fig = go.Figure()
feat_colors = {'danceability':'#1DB954','energy':'#FF6B6B','valence':'#FFD93D','acousticness':'#6BCB77'}
for feat, color in feat_colors.items():
    fig.add_trace(go.Bar(
        name=feat.capitalize(),
        x=genre_avg['genre_group'],
        y=genre_avg[feat],
        marker_color=color
    ))
fig.update_layout(
    barmode='group',
    title=dict(text='🎸 Average Audio Features by Genre Group', font_size=18),
    height=480,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor=SPOTIFY_BLACK,
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    xaxis=dict(gridcolor='#333', color=SPOTIFY_WHITE),
    yaxis=dict(gridcolor='#333', color=SPOTIFY_WHITE, range=[0,1])
)
fig.show()

In [14]:
# Energy vs Valence Scatter
fig = px.scatter(
    df,
    x='energy',
    y='valence',
    color='genre_group',
    size='popularity',
    hover_name='song_title',
    hover_data=['artist', 'primary_genre'],
    color_discrete_sequence=PALETTE,
    title='⚡ Energy vs Valence — All 135 Songs (Size = Popularity)',
    labels={'energy':'Energy','valence':'Valence (Positivity)','genre_group':'Genre'},
    size_max=25
)
fig.add_hline(y=0.5, line_dash='dot', line_color='#555')
fig.add_vline(x=0.5, line_dash='dot', line_color='#555')
fig.update_layout(
    height=540,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor='#1a1a2e',
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    title_font_size=18,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    xaxis=dict(gridcolor='#2a2a2a', color=SPOTIFY_WHITE, range=[0,1]),
    yaxis=dict(gridcolor='#2a2a2a', color=SPOTIFY_WHITE, range=[0,1])
)
fig.show()

In [15]:
# KMeans Clustering
FEATURE_COLS = ['danceability', 'energy', 'valence', 'acousticness', 'bpm_norm']
scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS])
inertias    = []
sil_scores  = []
K_range     = range(2, 11)
for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbl))
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Elbow Method (Inertia)', 'Silhouette Score']
)
fig.add_trace(go.Scatter(
    x=list(K_range), y=inertias,
    mode='lines+markers',
    line=dict(color=SPOTIFY_GREEN, width=3),
    marker=dict(size=9, color=SPOTIFY_GREEN)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=list(K_range), y=sil_scores,
    mode='lines+markers',
    line=dict(color='#FF6B6B', width=3),
    marker=dict(size=9, color='#FF6B6B')
), row=1, col=2)
fig.update_layout(
    title=dict(text='📐 Finding Optimal Number of Clusters', font_size=18),
    height=420,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor=SPOTIFY_BLACK,
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    showlegend=False
)
for axis in ['xaxis','xaxis2','yaxis','yaxis2']:
    fig.layout[axis].update(gridcolor='#333', color=SPOTIFY_WHITE)
fig.show()
print(f'Best Silhouette Score: {max(sil_scores):.3f} at K={list(K_range)[sil_scores.index(max(sil_scores))]}')

Best Silhouette Score: 0.328 at K=2


In [16]:
# Train final KMeans Model with K = 5
OPTIMAL_K = 5
kmeans        = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURE_COLS)
print('Cluster Centers:')
print(cluster_centers.round(3))
MOOD_LABELS = {
    0: '⚡ High Energy Bangers',
    1: '😊 Happy & Danceable',
    2: '🌙 Chill & Acoustic',
    3: '🔥 Intense & Dark',
    4: '💫 Feel-Good Pop'
}
centers_df = cluster_centers.copy()
centers_df['cluster_id'] = range(OPTIMAL_K)
centers_df = centers_df.sort_values('energy', ascending=False).reset_index(drop=True)
auto_labels = [
    '⚡ High Energy Bangers',
    '🔥 Intense & Dark',
    '😊 Happy & Danceable',
    '💫 Feel-Good Pop',
    '🌙 Chill & Acoustic'
]
cluster_name_map = dict(zip(centers_df['cluster_id'].values, auto_labels))
df['mood'] = df['cluster'].map(cluster_name_map)
print(f'\nSongs per mood cluster:')
print(df['mood'].value_counts())

Cluster Centers:
   danceability  energy  valence  acousticness  bpm_norm
0         0.760   0.659    0.786         0.093     0.420
1         0.372   0.578    0.317         0.117     0.364
2         0.498   0.707    0.540         0.070     0.766
3         0.568   0.332    0.625         0.402     0.236
4         0.167   0.186    0.119         0.555     0.154

Songs per mood cluster:
mood
🔥 Intense & Dark         49
💫 Feel-Good Pop          31
😊 Happy & Danceable      23
⚡ High Energy Bangers    20
🌙 Chill & Acoustic       12
Name: count, dtype: int64


In [17]:
# Visualize Clusters with PCA (2D)
pca        = PCA(n_components=2, random_state=42)
X_pca      = pca.fit_transform(X_scaled)
df['pca1'] = X_pca[:, 0]
df['pca2'] = X_pca[:, 1]
print(f'Explained variance by 2 PCA components: {pca.explained_variance_ratio_.sum()*100:.1f}%')
fig = px.scatter(
    df,
    x='pca1', y='pca2',
    color='mood',
    hover_name='song_title',
    hover_data=['artist', 'primary_genre', 'energy', 'valence', 'danceability'],
    color_discrete_sequence=PALETTE,
    title='🎯 KMeans Clusters — 2D PCA View of All Songs',
    labels={'pca1': 'PCA Component 1', 'pca2': 'PCA Component 2', 'mood': 'Mood Cluster'}
)
fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.update_layout(
    height=560,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor='#1a1a2e',
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    title_font_size=18,
    legend=dict(bgcolor='rgba(0,0,0,0)', title_text='Mood Cluster'),
    xaxis=dict(gridcolor='#2a2a2a', color=SPOTIFY_WHITE),
    yaxis=dict(gridcolor='#2a2a2a', color=SPOTIFY_WHITE)
)
fig.show()

Explained variance by 2 PCA components: 81.3%


In [18]:
# Content-Based Recommendation Model
similarity_matrix = cosine_similarity(X_scaled)
print(f'Similarity matrix shape: {similarity_matrix.shape}')
print(f'Example: similarity between song 0 and song 1 = {similarity_matrix[0][1]:.4f}')
def recommend_songs(song_title, n=5, same_genre=False, same_mood=True):
    """
    Recommend songs similar to the given song_title.

    Parameters:
    -----------
    song_title  : str   — Name of the input song (case-insensitive partial match)
    n           : int   — Number of recommendations to return
    same_genre  : bool  — If True, restrict to same broad genre group
    same_mood   : bool  — If True, prefer songs from the same mood cluster

    Returns:
    --------
    DataFrame of top N recommended songs with similarity scores
    """
    matches = df[df['song_title'].str.lower().str.contains(song_title.lower())]
    if matches.empty:
        print(f'❌ Song "{song_title}" not found in dataset.')
        print('Available songs:', df['song_title'].tolist()[:10], '...')
        return None
    song_idx   = matches.index[0]
    song_info  = df.loc[song_idx]
    print(f'🎵 Input Song  : {song_info["song_title"]} — {song_info["artist"]}')
    print(f'   Genre       : {song_info["primary_genre"]}  |  Mood: {song_info["mood"]}')
    print(f'   Energy: {song_info["energy"]}  |  Valence: {song_info["valence"]}  |  Dance: {song_info["danceability"]}')
    print('-' * 60)
    sim_scores = list(enumerate(similarity_matrix[song_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != song_idx]
    if same_genre:
        sim_scores = [s for s in sim_scores
                      if df.loc[s[0], 'genre_group'] == song_info['genre_group']]
    if same_mood:
        mood_scores = [s for s in sim_scores
                       if df.loc[s[0], 'cluster'] == song_info['cluster']]
        if len(mood_scores) >= n:
            sim_scores = mood_scores
    top_n = sim_scores[:n]
    results = []
    for idx, score in top_n:
        row = df.loc[idx]
        results.append({
            'Rank'           : len(results) + 1,
            'Song'           : row['song_title'],
            'Artist'         : row['artist'],
            'Genre'          : row['primary_genre'],
            'Mood Cluster'   : row['mood'],
            'Similarity'     : round(score, 4),
            'Energy'         : row['energy'],
            'Valence'        : row['valence'],
            'Danceability'   : row['danceability'],
            'Source'         : row['source']
        })
    return pd.DataFrame(results)
print('✅ Recommendation function ready!')

Similarity matrix shape: (135, 135)
Example: similarity between song 0 and song 1 = 0.9809
✅ Recommendation function ready!


In [19]:
# Recommendation Results & Evaluation
recs1 = recommend_songs('Espresso', n=8)
recs1

🎵 Input Song  : Espresso — Sabrina Carpenter
   Genre       : Pop  |  Mood: 🔥 Intense & Dark
   Energy: 0.73  |  Valence: 0.96  |  Dance: 0.8
------------------------------------------------------------


,Rank,Song,Artist,Genre,Mood Cluster,Similarity,Energy,Valence,Danceability,Source
0,1,Espresso (Remix),Sabrina Carpenter & Jimin,Pop/K-Pop,🔥 Intense & Dark,0.9992,0.74,0.92,0.82,Wrapped 2025
1,2,Closer,The Chainsmokers ft. Halsey,EDM/Pop,🔥 Intense & Dark,0.9987,0.65,0.77,0.74,All-Time Top 100
2,3,Lovin On Me,Jack Harlow,Hip-Hop/Pop,🔥 Intense & Dark,0.9969,0.62,0.85,0.78,Wrapped 2025
3,4,Please Please Please,Sabrina Carpenter,Pop,🔥 Intense & Dark,0.9963,0.66,0.87,0.70,Wrapped 2025
4,5,Baile Inolvidable,KAROL G,Latin Pop,🔥 Intense & Dark,0.9960,0.72,0.82,0.80,Wrapped 2025
5,6,Shape of You,Ed Sheeran,Pop/Dancehall,🔥 Intense & Dark,0.9957,0.65,0.93,0.83,All-Time Top 100
6,7,Water,Tyla,Afrobeats,🔥 Intense & Dark,0.9940,0.70,0.80,0.82,Wrapped 2025
7,8,Calm Down,Rema & Selena Gomez,Afrobeats/Pop,🔥 Intense & Dark,0.9938,0.72,0.78,0.80,All-Time Top 100


In [21]:
recs2 = recommend_songs('Blinding Lights', n=8)
recs2

🎵 Input Song  : Blinding Lights — The Weeknd
   Genre       : Synth-Pop  |  Mood: ⚡ High Energy Bangers
   Energy: 0.8  |  Valence: 0.33  |  Dance: 0.51
------------------------------------------------------------


,Rank,Song,Artist,Genre,Mood Cluster,Similarity,Energy,Valence,Danceability,Source
0,1,Blinding Lights (Remix),The Weeknd & Rosalía,Synth-Pop,⚡ High Energy Bangers,0.9989,0.80,0.36,0.54,All-Time Top 100
1,2,As It Was,Harry Styles,Indie Pop,⚡ High Energy Bangers,0.9963,0.73,0.36,0.52,All-Time Top 100
2,3,Bliss,Muse,Alt Rock,⚡ High Energy Bangers,0.9839,0.82,0.44,0.44,All-Time Top 100
3,4,Starboy,The Weeknd ft. Daft Punk,Synth-Pop,⚡ High Energy Bangers,0.9782,0.79,0.50,0.68,All-Time Top 100
4,5,Stay,The Kid LAROI & Justin Bieber,Pop,⚡ High Energy Bangers,0.9734,0.80,0.60,0.59,All-Time Top 100
5,6,good 4 u,Olivia Rodrigo,Pop Punk,⚡ High Energy Bangers,0.9689,0.90,0.65,0.56,Wrapped 2025
6,7,Cruel Summer,Taylor Swift,Synth-Pop,⚡ High Energy Bangers,0.9612,0.70,0.57,0.55,Wrapped 2025
7,8,Lose Yourself,Eminem,Hip-Hop,⚡ High Energy Bangers,0.9545,0.62,0.26,0.66,All-Time Top 100


In [22]:
recs3 = recommend_songs('APT.', n=8, same_genre=False, same_mood=True)
recs3

🎵 Input Song  : APT. — ROSE & Bruno Mars
   Genre       : K-Pop/Pop  |  Mood: 🔥 Intense & Dark
   Energy: 0.79  |  Valence: 0.88  |  Dance: 0.86
------------------------------------------------------------


,Rank,Song,Artist,Genre,Mood Cluster,Similarity,Energy,Valence,Danceability,Source
0,1,Million Dollar Baby,Tommy Richman,R&B/Pop,🔥 Intense & Dark,0.9990,0.76,0.80,0.77,Wrapped 2025
1,2,Boom Boom Boom Boom,ROSE,K-Pop/Pop,🔥 Intense & Dark,0.9987,0.82,0.92,0.84,Wrapped 2025
2,3,Exes,Tate McRae,Pop,🔥 Intense & Dark,0.9981,0.72,0.72,0.80,Wrapped 2025
3,4,Tusa,KAROL G & Nicki Minaj,Reggaeton,🔥 Intense & Dark,0.9965,0.78,0.72,0.78,Wrapped 2025
4,5,Positions,Ariana Grande,R&B/Pop,🔥 Intense & Dark,0.9952,0.66,0.69,0.74,All-Time Top 100
5,6,Taste,Sabrina Carpenter,Pop,🔥 Intense & Dark,0.9943,0.70,0.82,0.78,Wrapped 2025
6,7,Kiss Me More,Doja Cat ft. SZA,Pop/R&B,🔥 Intense & Dark,0.9935,0.66,0.80,0.78,All-Time Top 100
7,8,Sunroof,Nicky Youre & Dazy,Indie Pop,🔥 Intense & Dark,0.9929,0.80,0.96,0.76,All-Time Top 100


In [20]:
# Visualize Similarity Scores
fig = px.bar(
    recs1.sort_values('Similarity'),
    x='Similarity',
    y='Song',
    orientation='h',
    color='Similarity',
    color_continuous_scale=[[0,'#535353'],[0.5,'#1DB954'],[1,'#00FF7F']],
    text='Similarity',
    hover_data=['Artist','Genre','Mood Cluster'],
    title='🎵 Songs Most Similar to "Espresso" by Sabrina Carpenter'
)
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(
    height=500,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor=SPOTIFY_BLACK,
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    title_font_size=16,
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False, color=SPOTIFY_WHITE, range=[0, 1.1]),
    yaxis=dict(color=SPOTIFY_WHITE)
)
fig.show()

In [23]:
# Radar Chart for Input Song vs Top 3 Recommendations
def plot_radar_comparison(song_title, recs_df, top_n=3):
    feats    = ['danceability', 'energy', 'valence', 'acousticness']
    colors   = [SPOTIFY_GREEN, '#FF6B6B', '#FFD93D', '#6BCB77']
    fig      = go.Figure()
    match = df[df['song_title'].str.lower().str.contains(song_title.lower())]
    if match.empty:
        return
    src  = df.loc[match.index[0]]
    vals = [src[f] for f in feats] + [src[feats[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=feats + [feats[0]],
        fill='toself', name=f'🎵 {src["song_title"][:20]} (Input)',
        line=dict(color=SPOTIFY_GREEN, width=3), opacity=0.9
    ))
    for i, (_, row) in enumerate(recs_df.head(top_n).iterrows()):
        rec_match = df[df['song_title'] == row['Song']]
        if rec_match.empty:
            continue
        r     = df.loc[rec_match.index[0]]
        vals2 = [r[f] for f in feats] + [r[feats[0]]]
        fig.add_trace(go.Scatterpolar(
            r=vals2, theta=feats + [feats[0]],
            fill='toself', name=row['Song'][:22],
            line=dict(color=colors[i+1], width=2), opacity=0.7
        ))
    fig.update_layout(
        polar=dict(
            bgcolor='#1a1a2e',
            radialaxis=dict(visible=True, range=[0,1], color=SPOTIFY_WHITE, gridcolor='#444'),
            angularaxis=dict(color=SPOTIFY_WHITE, gridcolor='#444')
        ),
        title=dict(text=f'🕸️ Audio Profile: "{song_title}" vs Top 3 Recommendations', font_size=16),
        paper_bgcolor=SPOTIFY_BLACK,
        font=dict(color=SPOTIFY_WHITE, family='Arial'),
        legend=dict(bgcolor='rgba(0,0,0,0)', x=1.05),
        height=520
    )
    fig.show()
plot_radar_comparison('Espresso', recs1)

In [24]:
# Similarity Distribution across all songs
upper_tri = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
fig = go.Figure(go.Histogram(
    x=upper_tri,
    nbinsx=40,
    marker_color=SPOTIFY_GREEN,
    opacity=0.85
))
fig.update_layout(
    title=dict(text='📊 Distribution of Cosine Similarity Scores (All Song Pairs)', font_size=16),
    xaxis=dict(title='Cosine Similarity', color=SPOTIFY_WHITE, gridcolor='#333'),
    yaxis=dict(title='Number of Song Pairs', color=SPOTIFY_WHITE, gridcolor='#333'),
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor=SPOTIFY_BLACK,
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    height=420
)
fig.show()
print(f'Average similarity : {upper_tri.mean():.4f}')
print(f'Max similarity     : {upper_tri.max():.4f}')
print(f'Min similarity     : {upper_tri.min():.4f}')

Average similarity : 0.8355
Max similarity     : 0.9998
Min similarity     : 0.0087


In [25]:
# Interactive Recommendation Demo
print('🎵 All songs available for recommendation:')
print('=' * 55)
for i, row in df[['song_title','artist','genre_group']].iterrows():
    print(f"{i+1:3}. {row['song_title'][:35]:<35} | {row['artist'][:25]:<25} | {row['genre_group']}")

🎵 All songs available for recommendation:
  1. Die With A Smile                    | Lady Gaga & Bruno Mars    | Pop
  2. APT.                                | ROSE & Bruno Mars         | K-Pop
  3. Espresso                            | Sabrina Carpenter         | Pop
  4. Please Please Please                | Sabrina Carpenter         | Pop
  5. Taste                               | Sabrina Carpenter         | Pop
  6. Birds Of A Feather                  | Billie Eilish             | Pop
  7. Not Like Us                         | Kendrick Lamar            | Hip-Hop
  8. Good Luck, Babe!                    | Chappell Roan             | Pop
  9. Beautiful Things                    | Benson Boone              | Pop
 10. Lose Control                        | Teddy Swims               | R&B/Soul
 11. Too Sweet                           | Hozier                    | Indie/Folk
 12. Lunch                               | Billie Eilish             | Pop
 13. Boom Boom Boom Boom                

In [26]:
MY_SONG   = 'Die With A Smile'
N_RECS    = 8
SAME_MOOD = True
my_recs = recommend_songs(MY_SONG, n=N_RECS, same_mood=SAME_MOOD)
if my_recs is not None:
    display(my_recs.style
        .background_gradient(subset=['Similarity'], cmap='Greens')
        .background_gradient(subset=['Energy'],     cmap='Reds')
        .background_gradient(subset=['Valence'],    cmap='YlOrBr')
        .format({'Similarity': '{:.4f}', 'Energy': '{:.2f}',
                 'Valence': '{:.2f}', 'Danceability': '{:.2f}'})
        .set_table_styles([{
            'selector': 'th',
            'props': [('background-color','#1DB954'),('color','black'),('font-weight','bold')]
        }])
    )
    plot_radar_comparison(MY_SONG, my_recs)

🎵 Input Song  : Die With A Smile — Lady Gaga & Bruno Mars
   Genre       : Pop  |  Mood: 🔥 Intense & Dark
   Energy: 0.68  |  Valence: 0.55  |  Dance: 0.72
------------------------------------------------------------


,Rank,Song,Artist,Genre,Mood Cluster,Similarity,Energy,Valence,Danceability,Source
0,1,Tusa,KAROL G & Nicki Minaj,Reggaeton,🔥 Intense & Dark,0.9898,0.78,0.72,0.78,Wrapped 2025
1,2,Senorita,Shawn Mendes & Camila Cabello,Latin Pop,🔥 Intense & Dark,0.9892,0.74,0.75,0.75,All-Time Top 100
2,3,Exes,Tate McRae,Pop,🔥 Intense & Dark,0.9891,0.72,0.72,0.80,Wrapped 2025
3,4,Pink Pony Club,Chappell Roan,Synth-Pop,🔥 Intense & Dark,0.9851,0.80,0.62,0.72,Wrapped 2025
4,5,Attention,Charlie Puth,Pop,🔥 Intense & Dark,0.9837,0.68,0.60,0.78,All-Time Top 100
5,6,Havana,Camila Cabello ft. Young Thug,Latin Pop,🔥 Intense & Dark,0.9837,0.58,0.61,0.76,All-Time Top 100
6,7,Levii's Jeans,Beyoncé ft. Post Malone,Country/Pop,🔥 Intense & Dark,0.9821,0.65,0.72,0.72,Wrapped 2025
7,8,Million Dollar Baby,Tommy Richman,R&B/Pop,🔥 Intense & Dark,0.9820,0.76,0.80,0.77,Wrapped 2025


In [27]:
# Final Summary
mood_summary = df.groupby('mood').agg(
    song_count   = ('song_title', 'count'),
    avg_energy   = ('energy', 'mean'),
    avg_valence  = ('valence', 'mean'),
    avg_dance    = ('danceability', 'mean'),
    top_song     = ('song_title', 'first')
).reset_index().sort_values('avg_energy', ascending=False)
fig = px.bar(
    mood_summary,
    x='song_count',
    y='mood',
    orientation='h',
    color='avg_energy',
    color_continuous_scale=[[0,'#535353'],[0.5,'#1DB954'],[1,'#FF6B6B']],
    text='song_count',
    hover_data=['avg_energy','avg_valence','avg_dance','top_song'],
    title='🎭 Songs per Mood Cluster — Final Distribution',
    labels={'song_count':'Number of Songs','mood':'','avg_energy':'Avg Energy'}
)
fig.update_traces(texttemplate='%{text} songs', textposition='outside')
fig.update_layout(
    height=420,
    paper_bgcolor=SPOTIFY_BLACK,
    plot_bgcolor=SPOTIFY_BLACK,
    font=dict(color=SPOTIFY_WHITE, family='Arial'),
    title_font_size=16,
    coloraxis_showscale=False,
    xaxis=dict(showgrid=False, color=SPOTIFY_WHITE),
    yaxis=dict(color=SPOTIFY_WHITE)
)
fig.show()

In [43]:
# Streamlit UI
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
st.set_page_config(page_title="Spotify Recommender", layout="wide")
st.title("🎵 Spotify Intelligence Dashboard")

# LOAD DATA
@st.cache_data
def load_data():
    songs = pd.read_csv("Spotify Wrapped Top 50 Songs 2025.csv")
    artists = pd.read_csv("Spotify Wrapped Top 50 Artists 2025.csv")
    alltime = pd.read_csv("Spotify Alltime Top 100 Songs.csv")
    # Add dataset labels
    songs["dataset"] = "songs"
    artists["dataset"] = "artists"
    alltime["dataset"] = "alltime"
    df = pd.concat([songs, artists, alltime], ignore_index=True)
    # Clean column names
    df.columns = df.columns.str.strip().str.lower()
    return df
df = load_data()

# HANDLE COLUMN NAME VARIANTS
rename_map = {
    'track_name': 'song_title',
    'track': 'song_title',
    'artist_name': 'artist'
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
df = df.loc[:, ~df.columns.duplicated()]

# USE ONLY SONG DATA FOR ML
if 'dataset' in df.columns:
    df_model = df[df['dataset'] == 'songs'].copy()
else:
    df_model = df.copy()

# FEATURE ENGINEERING
FEATURE_COLS = ['danceability', 'energy', 'valence', 'acousticness']
FEATURE_COLS = [col for col in FEATURE_COLS if col in df_model.columns]
if len(FEATURE_COLS) == 0:
    st.error("No valid feature columns found")
    st.stop()
# Convert to numeric
for col in FEATURE_COLS:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')
# Fill missing values
df_model[FEATURE_COLS] = df_model[FEATURE_COLS].fillna(df_model[FEATURE_COLS].median())
# Final safety check
if df_model[FEATURE_COLS].isna().sum().sum() > 0:
    st.error("NaN values still present!")
    st.stop()
# Scale
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df_model[FEATURE_COLS])

# SIMILARITY
similarity_matrix = cosine_similarity(X_scaled)
def recommend_songs(song_title, n=5):
    if 'song_title' not in df_model.columns:
        return None
    matches = df_model[df_model['song_title'].astype(str).str.lower().str.contains(song_title.lower())]
    if matches.empty:
        return None
    idx = matches.index[0]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    recs = []
    for i, score in scores:
        recs.append({
            "Song": df_model.iloc[i].get('song_title', 'Unknown'),
            "Artist": df_model.iloc[i].get('artist', 'Unknown'),
            "Similarity": round(score, 4)
        })
    return pd.DataFrame(recs)

# SIDEBAR
st.sidebar.header("🔍 Controls")
song_input = st.sidebar.text_input("Enter Song Name", "Blinding Lights")
num_recs = st.sidebar.slider("Number of Recommendations", 3, 10, 5)

# TABS
tab1, tab2, tab3 = st.tabs(["📊 Overview", "🎯 Clustering", "🎵 Recommender"])

# TAB 1 — OVERVIEW
with tab1:
    st.subheader("Dataset Preview")
    st.dataframe(df.head())
    if 'energy' in df.columns:
        st.subheader("Feature Distribution")
        fig = px.histogram(df, x="energy")
        st.plotly_chart(fig, use_container_width=True)

# TAB 2 — CLUSTERING
with tab2:
    st.subheader("KMeans Clustering")
    k = st.slider("Select K", 2, 10, 5)
    try:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        df_model['cluster'] = kmeans.fit_predict(X_scaled)
        if 'energy' in df_model.columns and 'valence' in df_model.columns:
            fig = px.scatter(
                df_model,
                x="energy",
                y="valence",
                color=df_model['cluster'].astype(str),
                hover_name=df_model.get("song_title", None)
            )
            st.plotly_chart(fig, use_container_width=True)
    except Exception as e:
        st.error(f"Clustering failed: {e}")

# TAB 3 — RECOMMENDER
with tab3:
    st.subheader("🎵 Song Recommendation System")
    if st.button("Recommend"):
        recs = recommend_songs(song_input, num_recs)
        if recs is None or recs.empty:
            st.error("Song not found")
        else:
            st.success(f"Top {num_recs} recommendations:")
            st.dataframe(recs)
            fig = px.bar(recs, x="Similarity", y="Song", orientation='h')
            st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [44]:
# Run Streamlit
from pyngrok import ngrok
import subprocess
import time
import os
ngrok.kill()
PORT = 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
public_url = ngrok.connect(PORT)
print("🚀 Streamlit Public URL:", public_url)

🚀 Streamlit Public URL: NgrokTunnel: "https://4932-34-125-56-39.ngrok-free.app" -> "http://localhost:8501"
